In [ ]:
import os
import torch

from stark_qa import load_skb
from stark_neo4j_loading import insert_nodes, insert_relationships, insert_node_embeddings

from neo4j import GraphDatabase
from dotenv import load_dotenv
from tqdm import tqdm

%load_ext autoreload
%autoreload 2

#load neo4j credentials
load_dotenv('../db.env', override=True)
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_URI

In [4]:
DATASET_NAME = 'prime'

In [ ]:
from stark_qa import load_qa
qa = load_qa(DATASET_NAME)

In [74]:
from datasets import DatasetDict, Dataset
qa_dict = DatasetDict({split1: Dataset.from_pandas(qa.data.iloc[qa.get_idx_split()[split0].tolist()]) for split0, split1 in {'train':'train', 'val':'valid', 'test':'test'}.items()}) \
    .map(lambda x: x | {'answer_ids': eval(x['answer_ids'])}) \
    .rename_column('query', 'question')

Map:   0%|          | 0/5910 [00:00<?, ? examples/s]

Map:   0%|          | 0/1548 [00:00<?, ? examples/s]

Map:   0%|          | 0/1642 [00:00<?, ? examples/s]

In [79]:
qa_dict.save_to_disk(f'../{DATASET_NAME}-data/qa')

Saving the dataset (0/1 shards):   0%|          | 0/5910 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1548 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1642 [00:00<?, ? examples/s]

In [ ]:
# from datasets import load_from_disk
# qa_dict = load_from_disk('../amazon-data/qa')
# i=0
# 
# qa_dict['train']['question'][i], qa_dict['train']['answer_ids'][i]

In [ ]:
# DATASET_NAME = 'mag'
skb = load_skb(DATASET_NAME, download_processed=True, root=None)

In [ ]:
# Load pre-generated openai text-embedding-ada-002 embeddings
# Get emb_download.py from https://github.com/snap-stanford/stark. see Readme for other ways to generate embeddings
!python emb_download.py --dataset prime --emb_dir .

In [10]:
if DATASET_NAME=='amazon':
    #Turn reviews and QAs into strings and nested properties into direct node properties
    print("====== Preprocessing ======")
    for node_id, node in tqdm(skb.node_info.items()):
        if 'review' in node.keys():
            node['review'] = [str(review) for review in node['review']]
        if 'qa' in node.keys():
            node['qa'] = [str(qa) for qa in node['qa']]
        if 'details' in node.keys() and type(node['details']) is dict:
            for key, value in node['details'].items():
                node[key] = value
            del node['details']
    
    with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD)) as driver:
        insert_nodes(skb, driver=driver, key_property=None)
        insert_relationships(skb=skb, driver=driver, dataset_name=DATASET_NAME)
        
        product_description_embeddings = torch.load('./amazon/text-embedding-ada-002/doc/candidate_emb_dict.pt', weights_only=False)
        insert_node_embeddings(embeddings=product_description_embeddings, embedding_name='productDescriptionEmbedding', driver=driver)

if DATASET_NAME=='mag':
    #Rename attribute
    print("====== Preprocessing ======")
    for node_id, node in tqdm(skb.node_info.items()):
        if 'ConferenceSeriesId.1' in node.keys():
            node['ConferenceSeriesId1'] = node.pop('ConferenceSeriesId.1')
        
    with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD)) as driver:
        insert_nodes(skb, driver=driver, key_property='new_id') 
        insert_relationships(skb=skb, driver=driver, dataset_name=DATASET_NAME)

        abstract_embeddings = torch.load('./mag/text-embedding-ada-002/doc/candidate_emb_dict.pt', weights_only=False)
        insert_node_embeddings(embeddings=abstract_embeddings, embedding_name='abstractEmbedding', driver=driver)

        #Abstract embedding is used instead of title/name embedding for paper nodes.
        
        author_name_embeddings = torch.load('mag/text-embedding-ada-002/doc/author_name_embeddings.pt', weights_only=False)
        insert_node_embeddings(embeddings=author_name_embeddings, embedding_name='nameEmbedding', driver=driver)
        
        institution_name_embeddings = torch.load('./mag/text-embedding-ada-002/doc/institution_name_emb_dict.pt', weights_only=False)
        insert_node_embeddings(embeddings=institution_name_embeddings, embedding_name='nameEmbedding', driver=driver)
        
        field_of_study_name_embeddings = torch.load('./mag/text-embedding-ada-002/doc/field_of_study_name_emb_dict.pt', weights_only=False)
        insert_node_embeddings(embeddings=field_of_study_name_embeddings, embedding_name='nameEmbedding', driver=driver)
        
        #create vector and regular indexes
        
if DATASET_NAME=='prime':
    #Convert details (dict) to string
    print("====== Preprocessing ======")
    for node_id, node in tqdm(skb.node_info.items()):
        if 'details' in node.keys():
            node['details'] = str(node['details'])
    
    with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD)) as driver:
        insert_nodes(skb, driver=driver, key_property=None)
        insert_relationships(skb=skb, driver=driver, dataset_name=DATASET_NAME)
        
        text_embeddings = torch.load('./prime/text-embedding-ada-002/doc/candidate_emb_dict.pt', weights_only=False)
        insert_node_embeddings(embeddings=text_embeddings, embedding_name='textEmbedding', driver=driver)
        
        name_embeddings = torch.load('./prime/text-embedding-ada-002/doc/name_embeddings.pt', weights_only=False)
        insert_node_embeddings(embeddings=name_embeddings, embedding_name='nameEmbedding', driver=driver)

====== Preprocessing ======
======  Loading embeddings (nameEmbedding) ======


100%|██████████| 1021546/1021546 [50:18<00:00, 338.47it/s]


In [ ]:
# CREATE VECTOR INDEX abstractEmbedding IF NOT EXISTS
# FOR (p:Paper)
# ON p.abstract
# OPTIONS { indexConfig: {
#     `vector.dimensions`: 1536,
#     `vector.similarity_function`: 'cosine'
# }}